# Autism Screening Classification

This notebook is the portfolio-friendly walkthrough for the repository pipeline. It uses the same helper modules as `train.py`.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.config import (
    DATA_PATH,
    EXPERIMENT_NAME,
    N_ESTIMATORS,
    RANDOM_STATE,
    TARGET_COLUMN,
    TEST_SIZE,
)
from src.data.load_data import load_dataset
from src.evaluation.metrics import evaluate_model
from src.features.preprocessing import build_preprocessor, split_features_target
from src.models.train_model import build_model, build_pipeline, train_pipeline
from src.tracking.mlflow_utils import configure_mlflow, log_training_run

## 2. Config

In [ ]:
config_summary = {
    "data_path": str(DATA_PATH),
    "target_column": TARGET_COLUMN,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "n_estimators": N_ESTIMATORS,
    "experiment_name": EXPERIMENT_NAME,
}

pd.Series(config_summary)

## 3. Data Loading

In [ ]:
df = load_dataset(DATA_PATH)
df.head()

## 4. Quick EDA

In [ ]:
print("Dataset shape:", df.shape)
print("\nClass balance:")
print(df[TARGET_COLUMN].value_counts(normalize=True).rename("share"))

missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

## 5. Preprocessing

In [ ]:
X, y = split_features_target(df, target_col=TARGET_COLUMN)
preprocessor = build_preprocessor(X)

numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 7. Model Training

In [ ]:
model = build_model(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
pipeline = build_pipeline(preprocessor, model)
trained_pipeline = train_pipeline(pipeline, X_train, y_train)

trained_pipeline

## 8. Evaluation

In [ ]:
metrics = evaluate_model(trained_pipeline, X_test, y_test)
pd.DataFrame([metrics])

## 9. MLflow Logging

In [ ]:
tracking_uri = configure_mlflow(EXPERIMENT_NAME)
run_id = log_training_run(
    trained_pipeline,
    params={
        "model_type": "RandomForestClassifier",
        "n_estimators": N_ESTIMATORS,
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "target_column": TARGET_COLUMN,
    },
    metrics=metrics,
)

print("Tracking URI:", tracking_uri)
print("Run ID:", run_id)

## 10. Next Steps

- Compare this random forest baseline with a simple logistic regression model.
- Add feature importance analysis for model interpretation.
- Add lightweight unit tests for each helper module.
- Optionally connect DagsHub credentials to send MLflow runs to a remote tracker.